In [ ]:
!pip install -q openai openpyxl pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import time
import json
import re
import pandas as pd
from openai import OpenAI

# =============================================================================
# --- 1. CONFIGURATION ---
# =============================================================================

OPENROUTER_API_KEY = "....."  # new api by g7thesis

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODELS_TO_TEST = [
    "anthropic/claude-sonnet-4-6",
    "deepseek/deepseek-v3.2",
    "google/gemini-3.1-pro-preview",
    "openai/gpt-5.2-pro"
]

# max_tokens=4000 for ALL models.
# This is a ceiling, not a reservation — you are billed only for tokens
# actually generated. Claude finishing at 964 tokens with max_tokens=4000
# costs the same as with max_tokens=964. The flat 4000 ceiling:
#   - Guarantees Gemini never truncates (was hitting 2,586 ceiling before)
#   - Gives DeepSeek R1 room for <think> blocks (stripped before parsing)
#   - Has zero cost penalty for GPT and Claude which finish well under 2,000
MAX_OUTPUT_TOKENS = 5000

# Output paths — all on Google Drive so they survive Colab runtime resets
DRIVE_BASE       = "/content/drive/MyDrive/WBC Dataset/HPM Value Generate by LLM"
CSV_PARAMS_PATH  = f"{DRIVE_BASE}/HPO_LLM_params_live.csv"
CSV_METRICS_PATH = f"{DRIVE_BASE}/HPO_LLM_metrics_live.csv"
EXCEL_FINAL_PATH = f"{DRIVE_BASE}/FrontierModels_HPO_Results.xlsx"

# =============================================================================
# --- 2. INCREMENTAL SAVE HELPERS ---
# =============================================================================

def _append_rows_to_csv(rows: list[dict], path: str):
    """
    Append a list of dicts to a CSV file.
    Creates the file with headers on first write.
    Appends without headers on subsequent writes.
    Called after every successful combination AND on KeyboardInterrupt.
    """
    if not rows:
        return
    df_new     = pd.DataFrame(rows)
    file_exists = os.path.exists(path)
    df_new.to_csv(
        path,
        mode="a",
        header=not file_exists,
        index=False,
    )


def _resume_from_csv(path: str) -> tuple[set, int]:
    """
    Check for an existing partial-run CSV and return:
      - set of Combination_IDs already completed
      - next global_id to continue from

    Enables automatic resume after a crash or manual stop.
    """
    if not os.path.exists(path):
        return set(), 1
    try:
        df_existing = pd.read_csv(path)
        completed   = set(df_existing["Combination_ID"].unique())
        next_id     = int(df_existing["ID"].max()) + 1
        print(
            f"♻️  Resuming: {len(completed)} combinations already saved, "
            f"next ID = {next_id}"
        )
        return completed, next_id
    except Exception as e:
        print(f"⚠️  Could not read existing CSV ({e}). Starting fresh.")
        return set(), 1


def _build_final_excel():
    """
    Assemble the final Excel workbook from the two live CSVs.
    Called both at normal pipeline completion and on KeyboardInterrupt,
    so the Excel always reflects the latest saved state.
    """
    if not (os.path.exists(CSV_PARAMS_PATH) or os.path.exists(CSV_METRICS_PATH)):
        print("   No CSV data found — Excel not written.")
        return

    print(f"\n📦 Assembling Excel from saved CSVs → {EXCEL_FINAL_PATH}")
    with pd.ExcelWriter(EXCEL_FINAL_PATH, engine="openpyxl") as writer:

        if os.path.exists(CSV_PARAMS_PATH):
            df_params = pd.read_csv(CSV_PARAMS_PATH)
            df_params.to_excel(writer, sheet_name="Params", index=False)
            print(f"   Params  : {len(df_params)} rows")

            over_limit = df_params[df_params["logic_over_limit"] == True]
            if len(over_limit) > 0:
                print(f"   ⚠️  Logic over 100 words: {len(over_limit)} suggestions")
            else:
                print("   ✅ All logic strings within 100-word limit.")

        if os.path.exists(CSV_METRICS_PATH):
            df_metrics = pd.read_csv(CSV_METRICS_PATH)
            df_metrics.to_excel(writer, sheet_name="Metrics", index=False)
            success_count = (df_metrics["Status"] == "Success").sum()
            failed_count  = (df_metrics["Status"] != "Success").sum()
            print(f"   Metrics : {len(df_metrics)} rows "
                  f"({success_count} success, {failed_count} failed/incomplete)")

    print(f"   ✅ Excel saved → {EXCEL_FINAL_PATH}")


# =============================================================================
# --- 3. DATA LOADING ---
# =============================================================================

_LOG_FILE_PATH = "/content/drive/MyDrive/WBC Dataset/my_hpo_logs.xlsx"

try:
    _HPO_LOGS_DF: pd.DataFrame | None = pd.read_excel(_LOG_FILE_PATH)
    print(f"✅ HPO logs loaded into memory: {len(_HPO_LOGS_DF)} rows.")
except FileNotFoundError:
    _HPO_LOGS_DF = None
    print("⚠️  WARNING: HPO log file not found. RAG contexts will be unavailable.")


def get_rag_context(context_type: str) -> str:
    if context_type == "no_context":
        return (
            "No historical training logs are available. "
            "Reason entirely from your ML expertise and the experimental "
            "constraints described above."
        )
    if _HPO_LOGS_DF is None:
        return "Error: Log file was not loaded at startup. Cannot provide RAG context."

    df = _HPO_LOGS_DF

    if context_type == "best_values":
        if "val_accuracy" not in df.columns:
            return "Error: 'val_accuracy' column missing from log file."
        top_20 = df.sort_values(by="val_accuracy", ascending=False).head(20)
        return f"Top 20 performing historical trials:\n{top_20.to_markdown(index=False)}"

    if context_type == "all_logs":
        cols_to_keep  = ["learning_rate", "batch_size", "val_accuracy", "val_loss"]
        existing_cols = [c for c in cols_to_keep if c in df.columns]
        df_trimmed    = df[existing_cols].head(59)
        return f"Full historical training logs:\n{df_trimmed.to_markdown(index=False)}"

    return "Error: Unknown context_type provided."


# =============================================================================
# --- 4. JSON EXTRACTION & REPAIR ---
# =============================================================================

def extract_and_parse_json(text: str) -> dict:
    """
    Four-strategy JSON recovery before ever raising ValueError.
    Prevents token-wasting retries on repairable truncated responses.
    """

    # Strategy 1 — direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strategy 2 — extract outermost {...} block
    json_match = re.search(r"\{.*\}", text, re.DOTALL)
    candidate  = json_match.group(0) if json_match else text
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Strategy 3 — truncation repair
    try:
        complete_suggestions = re.findall(
            r'\{[^{}]*"logic"\s*:\s*"[^"]*"[^{}]*\}',
            candidate,
            re.DOTALL,
        )
        if complete_suggestions:
            repaired = '{"suggestions": [' + ",".join(complete_suggestions) + "]}"
            result   = json.loads(repaired)
            print(
                f"         ⚠️  JSON truncated — recovered "
                f"{len(result.get('suggestions', []))} of 5 suggestions."
            )
            return result
    except Exception:
        pass

    # Strategy 4 — aggressive clean
    try:
        cleaned = re.sub(r"```(?:json)?|```", "", candidate)
        cleaned = re.sub(r",\s*([}\]])", r"\1", cleaned)
        cleaned = (
            cleaned
            .replace("True",  "true")
            .replace("False", "false")
            .replace("None",  "null")
            .strip()
        )
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    raise ValueError(
        f"All JSON recovery strategies failed. Preview: {text[:200]}"
    )


# =============================================================================
# --- 5. PROMPT ENGINEERING ---
# =============================================================================

def build_prompts(
    method: str,
    rag_context_str: str,
    rag_context_type: str,
) -> tuple[str, str, str]:

    effective_method = method
    if method == "generated_knowledge" and rag_context_type == "no_context":
        effective_method = "theoretical_anchoring"

    system_prompt = (
        "You are an ML Research Scientist specializing in HPO for Medical Imaging.\n\n"

        "TASK: Suggest 5 distinct (learning_rate, batch_size) sets for an "
        "EfficientNetB3 WBC classifier.\n\n"

        "SETUP:\n"
        "- Dataset    : 16,027 WBC images. Imbalance: min=99, max=3,046 (~31:1). "
        "Class weights 0–10 applied per sample.\n"
        "- Model      : EfficientNetB3 → GlobalAvgPool → Dropout(0.5) → Dense(softmax). "
        "ALL BatchNorm layers FROZEN; only Dense head trains.\n"
        "- Training   : Gradient clipping max_norm=1.0. "
        "Fixed random 80/10/10 train/val/test split.\n"
        "- Objective  : Maximise val_accuracy, minimise val_loss.\n\n"

        "Return ONLY valid JSON — no prose, no fences:\n"
        "{\n"
        '  "suggestions": [{\n'
        '    "learning_rate": <float>,\n'
        '    "batch_size": <int>,\n'
        '    "predicted_val_accuracy": <float 0–1, your confidence this set works well>,\n'
        '    "predicted_val_loss": <float, expected error proportion — '
        'must be consistent: higher confidence → lower error>,\n'
        '    "logic": "<100-WORD MAX: why this LR and BS, grounded in '
        'frozen BN, 31:1 imbalance, weights, grad clipping — no generic filler>"\n'
        "  }]\n"
        "}"
    )

    method_instructions = {

        "zero_shot": (
            "APPROACH: ZERO-SHOT\n"
            "Use only your ML expertise and the constraints above.\n"
            "Each logic must causally connect frozen BN, the 99-sample minority "
            "class with 10x weights, and grad clipping to your chosen LR and BS."
        ),

        "chain_of_thought": (
            "APPROACH: CHAIN-OF-THOUGHT\n"
            "Reason through these steps, then output JSON:\n"
            "  1. What does freezing all BN layers imply for gradient flow "
            "and feature adaptation?\n"
            "  2. With min_class=99 and weights up to 10x, what is the minimum "
            "BS guaranteeing minority representation per batch? How does "
            "amplified minority gradient affect LR stability?\n"
            "  3. How does grad clipping at 1.0 set the upper bound of viable LR?\n"
            "  4. If logs provided: find Pareto-optimal region. "
            "If not: build a search grid from steps 1–3.\n"
            "  5. Pick 5 points spanning conservative, moderate, and exploratory regions."
        ),

        "generated_knowledge": (
            "APPROACH: EVIDENCE-ANCHORED\n"
            "1. Extract 3 heuristics from logs. "
            "Format: 'Observation: [X]. Rule: [Y for LR or BS].'\n"
            "2. Discard any candidate that violates a rule.\n"
            "3. Output JSON with only rule-compliant suggestions."
        ),

        "theoretical_anchoring": (
            "APPROACH: THEORY-ANCHORED\n"
            "No logs available — derive heuristics from ML theory.\n"
            "1. Derive 3 heuristics specific to this experiment "
            "(frozen BN + 31:1 imbalance + weights 0–10 + grad clip + head-only training). "
            "Format: 'Principle: [theory]. Rule: [Y for LR or BS].'\n"
            "2. Discard any candidate that violates a rule.\n"
            "3. Output JSON with only rule-compliant suggestions."
        ),
    }

    user_prompt = (
        f"**HISTORICAL TRIALS (RAG CONTEXT):**\n"
        f"{rag_context_str}\n\n"
        f"{method_instructions[effective_method]}"
    )

    return system_prompt, user_prompt, effective_method


# =============================================================================
# --- 6. API PARAMETER BUILDER ---
# =============================================================================

def build_api_params(model_name: str, sys_prompt: str, usr_prompt: str) -> dict:
    """
    Flat MAX_OUTPUT_TOKENS=4000 for all models.
    Ceiling only — no cost for unused capacity.
    DeepSeek gets lower temperature; all others get response_format=json_object.
    """
    params = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user",   "content": usr_prompt},
        ],
        "max_tokens": MAX_OUTPUT_TOKENS,
    }

    if "deepseek" in model_name.lower():
        params["temperature"] = 0.6
    else:
        params["temperature"]     = 0.7
        params["response_format"] = {"type": "json_object"}

    return params


# =============================================================================
# --- 7. EXECUTION ENGINE ---
# =============================================================================

def run_pipeline():
    rag_types = ["no_context", "best_values", "all_logs"]
    methods   = ["zero_shot", "chain_of_thought", "generated_knowledge"]

    total_combos = len(MODELS_TO_TEST) * len(rag_types) * len(methods)
    run_counter  = 0

    # Buffers for the current in-progress combination.
    # Flushed to CSV on success OR on KeyboardInterrupt — whichever comes first.
    pending_params_rows  = []
    pending_metrics_rows = []

    completed_combos, global_id = _resume_from_csv(CSV_PARAMS_PATH)

    print("🚀 Starting HPO LLM Pipeline via OpenRouter...")
    print(f"   Models      : {len(MODELS_TO_TEST)}")
    print(f"   RAG types   : {len(rag_types)}")
    print(f"   Methods     : {len(methods)}")
    print(f"   Max tokens  : {MAX_OUTPUT_TOKENS} (all models)")
    print(f"   Total combos: {total_combos}")
    if completed_combos:
        print(f"   Skipping    : {len(completed_combos)} already completed")
        print(f"   Remaining   : {total_combos - len(completed_combos)}")
    print()

    try:
        for model_name in MODELS_TO_TEST:
            for rag in rag_types:
                for method in methods:

                    run_counter += 1
                    context = get_rag_context(rag)
                    sys_prompt, usr_prompt, effective_method = build_prompts(
                        method, context, rag
                    )
                    api_params = build_api_params(model_name, sys_prompt, usr_prompt)

                    combo_name = (
                        f"{model_name.split('/')[-1]} | {rag} | {effective_method}"
                    )

                    if combo_name in completed_combos:
                        print(
                            f"  [{run_counter:02d}/{total_combos}] "
                            f"⏭️  Skipping (already saved): {combo_name}"
                        )
                        continue

                    print(f"  [{run_counter:02d}/{total_combos}] {combo_name}")

                    # Reset pending buffers for this combination
                    pending_params_rows  = []
                    pending_metrics_rows = []

                    max_retries = 3
                    success     = False

                    for attempt in range(max_retries):
                        start_time = time.time()
                        try:
                            response = client.chat.completions.create(**api_params)
                            latency  = round(time.time() - start_time, 4)
                            tot_tok  = (
                                response.usage.total_tokens
                                if response.usage else 0
                            )

                            # Guard: empty choices
                            if not response.choices:
                                raise ValueError(
                                    "Empty choices list — likely context window exceeded. "
                                    f"Response: {response}"
                                )

                            # Guard: None content
                            raw_content = response.choices[0].message.content
                            if raw_content is None:
                                raise ValueError(
                                    f"content=None returned. Response: {response}"
                                )
                            text_res = raw_content.strip()

                            # Strip DeepSeek <think> tags
                            text_res = re.sub(
                                r"<think>.*?</think>", "", text_res, flags=re.DOTALL
                            ).strip()

                            # Robust JSON extraction
                            json_data   = extract_and_parse_json(text_res)
                            suggestions = json_data.get("suggestions", [])

                            if not suggestions:
                                raise ValueError(
                                    "JSON parsed but 'suggestions' array is empty."
                                )

                            # Build rows into pending buffers
                            for i, item in enumerate(suggestions):
                                logic_text       = str(item.get("logic", ""))
                                logic_word_count = len(logic_text.split())

                                pending_params_rows.append({
                                    "ID":                     global_id,
                                    "Combination_ID":         combo_name,
                                    "Model":                  model_name.split("/")[-1],
                                    "RAG_Type":               rag,
                                    "Requested_Method":       method,
                                    "Effective_Method":       effective_method,
                                    "Suggestion_ID":          i + 1,
                                    "learning_rate":          item.get("learning_rate"),
                                    "batch_size":             item.get("batch_size"),
                                    "predicted_val_accuracy": item.get("predicted_val_accuracy"),
                                    "predicted_val_loss":     item.get("predicted_val_loss"),
                                    "logic":                  logic_text,
                                    "logic_word_count":       logic_word_count,
                                    "logic_over_limit":       logic_word_count > 100,
                                    "val_accuracy":           None,
                                    "val_loss":               None,
                                })
                                global_id += 1

                            pending_metrics_rows.append({
                                "Combination_ID":   combo_name,
                                "Model":            model_name.split("/")[-1],
                                "RAG_Type":         rag,
                                "Requested_Method": method,
                                "Effective_Method": effective_method,
                                "N_Suggestions":    len(suggestions),
                                "Latency_s":        latency,
                                "Total_Tokens":     tot_tok,
                                "Status":           "Success",
                            })

                            # Flush pending rows to CSV immediately
                            _append_rows_to_csv(pending_params_rows,  CSV_PARAMS_PATH)
                            _append_rows_to_csv(pending_metrics_rows, CSV_METRICS_PATH)
                            pending_params_rows  = []   # Clear after successful save
                            pending_metrics_rows = []

                            print(
                                f"         ✅ Attempt {attempt + 1} | "
                                f"Latency: {latency}s | "
                                f"Tokens: {tot_tok} | "
                                f"Suggestions: {len(suggestions)} | "
                                f"💾 Saved"
                            )
                            success = True
                            break

                        except Exception as e:
                            if attempt < max_retries - 1:
                                wait = 2 ** attempt
                                print(
                                    f"         ⚠️  Attempt {attempt + 1} failed: {e}. "
                                    f"Retrying in {wait}s..."
                                )
                                time.sleep(wait)
                            else:
                                print(
                                    f"         ❌ Failed after {max_retries} attempts: {e}"
                                )
                                _append_rows_to_csv([{
                                    "Combination_ID":   combo_name,
                                    "Model":            model_name.split("/")[-1],
                                    "RAG_Type":         rag,
                                    "Requested_Method": method,
                                    "Effective_Method": effective_method,
                                    "Status":           f"FAILED: {e}",
                                }], CSV_METRICS_PATH)

                    if success:
                        time.sleep(4)

    except KeyboardInterrupt:
        # ── GRACEFUL STOP HANDLER ──────────────────────────────────────────
        # Triggered when you press the Stop button in Colab or Ctrl+C.
        # At this point pending_params_rows and pending_metrics_rows contain
        # the in-progress combination's data if the API call had already
        # returned successfully but the CSV flush was interrupted, OR they
        # are empty if the stop happened mid-API-call (in which case there
        # is nothing to save for that combination).
        print("\n\n⛔ Pipeline stopped manually.")

        if pending_params_rows:
            print(
                f"   💾 Flushing {len(pending_params_rows)} in-progress rows to CSV..."
            )
            _append_rows_to_csv(pending_params_rows,  CSV_PARAMS_PATH)
            _append_rows_to_csv(pending_metrics_rows, CSV_METRICS_PATH)
            print("   ✅ In-progress rows saved.")
        else:
            print(
                "   ℹ️  No in-progress rows to flush "
                "(stop occurred mid-API-call — current combination not saved)."
            )

    finally:
        # ── ALWAYS runs — whether normal finish, KeyboardInterrupt, or crash
        # Builds the Excel from whatever is currently in the CSVs.
        # Even a mid-run stop produces a valid Excel of all completed work.
        _build_final_excel()
        print("\n✅ Pipeline exited cleanly.")
        print(f"   Live params CSV  → {CSV_PARAMS_PATH}")
        print(f"   Live metrics CSV → {CSV_METRICS_PATH}")
        print(f"   Final Excel      → {EXCEL_FINAL_PATH}")


# =============================================================================
# --- 8. ENTRY POINT ---
# =============================================================================

# if __name__ == "__main__":
#     run_pipeline()

# For Colab — uncomment if __name__ guard does not trigger:
run_pipeline()

✅ HPO logs loaded into memory: 59 rows.
🚀 Starting HPO LLM Pipeline via OpenRouter...
   Models      : 4
   RAG types   : 3
   Methods     : 3
   Max tokens  : 5000 (all models)
   Total combos: 36

  [01/36] claude-sonnet-4-6 | no_context | zero_shot
         ✅ Attempt 1 | Latency: 16.4782s | Tokens: 1271 | Suggestions: 5 | 💾 Saved
  [02/36] claude-sonnet-4-6 | no_context | chain_of_thought
         ✅ Attempt 1 | Latency: 15.1567s | Tokens: 1365 | Suggestions: 5 | 💾 Saved
  [03/36] claude-sonnet-4-6 | no_context | theoretical_anchoring
         ✅ Attempt 1 | Latency: 22.0992s | Tokens: 1577 | Suggestions: 5 | 💾 Saved
  [04/36] claude-sonnet-4-6 | best_values | zero_shot
         ✅ Attempt 1 | Latency: 16.3335s | Tokens: 2775 | Suggestions: 5 | 💾 Saved
  [05/36] claude-sonnet-4-6 | best_values | chain_of_thought
         ✅ Attempt 1 | Latency: 15.496s | Tokens: 2828 | Suggestions: 5 | 💾 Saved
  [06/36] claude-sonnet-4-6 | best_values | generated_knowledge
         ⚠️  JSON truncated — 